# Static linear encoding and Information Imbalance

This notebook combines bidirectional static linear encoding with Information Imbalance (II) at one neural-response timepoint. It performs the following operations:

1. Predict model features from the selected neural population response (`neural -> model`).
2. Predict the neural population response from model features (`model -> neural`).
3. Compute per-output $R^2$ for both projections.
4. Compute static II between the **predicted neural** and **predicted model** spaces.

By default, the projections are fitted and evaluated on all images (`use_cv=False`). Setting `use_cv=True` produces out-of-fold predictions, and II is then computed only from those out-of-fold predictions.

In [1]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from sklearn.decomposition import PCA
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import ImageFolder
from transformers import AutoImageProcessor
import math

# Locate the repository whether Jupyter starts in the project root or scripts folder.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate config.yaml from the current directory.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])

# from II_analyses.static_encoding import (
#     compute_static_prediction_II,
#     fit_static_projection,
# )
from project_specific_utils.dataloader import (
    load_img_natraster,
    map_image_order_from_ann_to_monkey,
)
from IT_recap.hf_feature_extraction import (
    ProcessorTransform,
    is_valid_image_file,
)
from useful_stuff.general_utils.regression import linear_encoding
from useful_stuff.image_processing.computational_models import imgANN


In [2]:
@dataclass
class Cfg:
    # Neural recording and image-set parameters.
    monkey_name: str = "three0"
    date: str = "250313"
    brain_area: str = "AIT"
    folder_name: str = "talia_20each_tizi"
    new_fs: int = 100
    # pca_comp = 100
    # Static model-feature parameters.
    model_name: str = "dino_v3_l"
    model_source: str = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    img_size: int = 224
    use_fast_processor: bool = True
    pkg = "hf"
    layer_names: list[str] = field(default_factory=lambda: [
        "layer.3.mlp.down_proj",
        "layer.13.mlp.down_proj",
        "layer.20.mlp.down_proj",
    ])
    pooling: str = "mean"
    trust_remote_code = True
    attn_implementation = "sdpa"

    batch_size = 8
    # # Linear encoding parameters. CV is deliberately disabled for this run.
    # regression_type: str = "lr"
    # use_cv: bool = False
    # cv_type: str = "same"
    # n_splits: int = 5
    # shuffle: bool = True
    # random_seed: int = 0
    # alpha_min: float = 1e-6
    # alpha_max: float = 1e3
    # n_alphas: int = 10

    # # Static Information Imbalance parameters.
    # neural_RDM_metric: str = "cosine_cnt"
    # model_RDM_metric: str = "cosine_cnt"
    # k: int = 10
# EOC


cfg = Cfg()
cfg


Cfg(monkey_name='three0', date='250313', brain_area='AIT', folder_name='talia_20each_tizi', new_fs=100, model_name='dino_v3_l', model_source='facebook/dinov3-vitl16-pretrain-lvd1689m', img_size=224, use_fast_processor=True, layer_names=['layer.3.mlp.down_proj', 'layer.13.mlp.down_proj', 'layer.20.mlp.down_proj'], pooling='mean')

## Load and align the static spaces

The neural raster is shaped `(neurons, timepoints, images)`. Model features are reordered to the monkey image-presentation order and remain shaped `(model features, images)`.

In [3]:
raster = load_img_natraster(
    paths,
    cfg.monkey_name,
    cfg.date,
    new_fs=cfg.new_fs,
    brain_area=cfg.brain_area,
)

# Load the preprocessing recipe saved with the Hugging Face checkpoint.
image_processor = AutoImageProcessor.from_pretrained(
    cfg.model_source,
    use_fast=cfg.use_fast_processor,
)

dataset_path = Path(paths["livingstone_lab"]) / "Stimuli" / cfg.folder_name
dataset = ImageFolder(
    root=dataset_path,
    transform=ProcessorTransform(image_processor),
    is_valid_file=is_valid_image_file,
    allow_empty=True,
)

# Fail early if the checkpoint processor and configured model size disagree.
sample_image, _ = dataset[0]
sample_shape = tuple(sample_image.shape[-2:])
if sample_shape != (cfg.img_size, cfg.img_size):
    raise ValueError(
        f"Processor returned {sample_shape}, expected "
        f"{(cfg.img_size, cfg.img_size)}."
    )
# end if processor output size differs from model input size
idx_ord = map_image_order_from_ann_to_monkey(
    paths, cfg.monkey_name, cfg.date, dataset
)

# features_path = (
#     Path(paths["data_path"])
#     / "models"
#     / (
#         f"{cfg.folder_name}_{cfg.model_name}_{cfg.img_size}_{cfg.layer_name}"
#         f"_features_{cfg.pooling}pool.npz"
#     )
# )
# model_features = np.load(features_path)["arr_0"][:, idx_ord]
# if cfg.pca_comp is not None:
#     pca_obj = PCA(n_components=cfg.pca_comp)
#     model_features = pca_obj.fit_transform(model_features.T)
#     model_features = model_features.T

    
raster_array = raster.get_array()

if raster_array.ndim != 3:
    raise ValueError(
        "Expected neural data with shape (neurons, timepoints, images), "
        f"received {raster_array.shape}."
    )
# end if raster_array.ndim != 3
# if model_features.ndim != 2:
#     raise ValueError(
#         "Expected model data with shape (features, images), "
#         f"received {model_features.shape}."
#     )
# end if model_features.ndim != 2
# if raster_array.shape[2] != model_features.shape[1]:
#     raise ValueError(
#         "Neural and model spaces have different image counts: "
#         f"{raster_array.shape[2]} and {model_features.shape[1]}."
#     )
# # end if raster_array.shape[2] != model_features.shape[1]

print(f"Neural raster: {raster_array.shape}")
# print(f"Aligned model features: {model_features.shape}")
# print(f"Model features loaded from: {features_path}")


Neural raster: (25, 30, 776)


In [4]:
class NeuralImageDataset(Dataset):
    """Pair images with neural activity in the same stimulus order."""

    def __init__(self, image_dataset, neural_activity):
        # Expected input: [neurons, time, images].
        neural_activity = np.asarray(neural_activity)

        if len(image_dataset) != neural_activity.shape[2]:
            raise ValueError(
                f"Found {len(image_dataset)} images but "
                f"{neural_activity.shape[2]} neural responses."
            )
        # end if image and response counts differ

        self.image_dataset = image_dataset

        # Store targets as [images, time, neurons].
        self.neural_activity = torch.as_tensor(
            neural_activity.transpose(2, 1, 0),
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.image_dataset)
    # EOF

    def __getitem__(self, index):
        image, _ = self.image_dataset[index]
        target = self.neural_activity[index]  # [time, neurons]
        return image, target
    # EOF
# EOC


In [5]:
ann = imgANN(
    model_name=cfg.model_name,
    pkg=cfg.pkg,
    img_size=cfg.img_size,
    pooling=cfg.pooling,
    dtype=torch.float32,
    attn_implementation=cfg.attn_implementation,
    repo_url=cfg.model_source,
    trust_remote_code=cfg.trust_remote_code,
)

15:46:49 - device being used: mps


In [6]:
ordered_image_dataset = Subset(dataset, idx_ord)

paired_dataset = NeuralImageDataset(
    image_dataset=ordered_image_dataset,
    neural_activity=raster_array,  # [neurons, time, images]
)

loader = DataLoader(
    paired_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    pin_memory=torch.device(ann.get_device()).type == "cuda",
)


# TODO 
1. import and load imgANN
2. make the forward pass work and comment that tidily
3. make the loss function (linear mapping and R^2)
4. make the training loop

In [7]:
class BaselineModel(torch.nn.Module):
    def __init__(
            self, 
            encoder, 
            layers, 
            temporal_embedding_dim, 
            value_dim, 
            n_timepoints, 
            temporal_compression_ratio, 
            key_query_dim=None
            ):    
        super(BaselineModel, self).__init__()
        self.layer_names = list(layers)
        self.n_layers = len(self.layer_names)

        # set up the encoder
        encoder.model.eval()
        self.encoder = encoder
        for parameter in self.encoder.model.parameters():
            parameter.requires_grad_(False)
        # end for parameter

        self.encoder.set_relevant_layers(layers)
        self.encoder_dim = self.encoder.get_layer_output_shape(layers[0])[1] # assumes all layers to have the same embedding dimension (Visual transformer case)
        self.encoder.create_forward_hook() # it's taking the relevant layers we've set

        # Register the underlying nn.Module as a real submodule so that
        # BaselineModel.to(device)/.parameters()/.state_dict() see its weights too.
        # `imgANN` itself is a plain wrapper (not an nn.Module), so assigning it to
        # self.encoder above does NOT register it - self.encoder.model IS the same
        # nn.Module object, so hooks registered on it by create_forward_hook() keep
        # working regardless of which attribute we reach it through.
        self.encoder_backbone = self.encoder.model

        # set up temporal embeddings
        # Learned coarse temporal queries:
        # (T_te, E_te)
        self.temporal_compression_ratio = temporal_compression_ratio
        self.n_temporal_embeddings = n_timepoints / temporal_compression_ratio
        self.n_timepoints = n_timepoints
        self.temporal_embedding_dim = temporal_embedding_dim
        self.temporal_embeddings = nn.Parameter(
            torch.randn(
                round(n_timepoints / temporal_compression_ratio), # TODO see how to treat the cases in which it doesn't yield an int
                self.temporal_embedding_dim
            )
            * 0.02
        ) # TODO check out how to initialize

        # set up attention dimensions
        if key_query_dim is not None:
            self.key_dim = key_query_dim
            self.query_dim = key_query_dim
        else:
            self.key_dim = self.temporal_embedding_dim
            self.query_dim = self.temporal_embedding_dim
        # end if key_query_dim is not None:
        self.value_dim = value_dim

        # Backbone layer features -> keys
        # (B, L, E_in) -> (B, L, E_k)
        self.key_projection = nn.Linear(
            self.encoder_dim,
            self.key_dim,
            bias=False,
        )
        # Backbone layer features -> values
        # (B, L, E_in) -> (B, L, E_v)
        self.value_projection = nn.Linear(
            self.encoder_dim,
            self.value_dim,
            bias=False,
        )
        if self.query_dim != self.temporal_embedding_dim:
            self.query_projection = nn.Linear(
                self.temporal_embedding_dim,
                self.value_dim,
                bias=False,
            )
            
        # normalization of projected layer features
        self.key_norm = nn.LayerNorm(self.key_dim)
        self.value_norm = nn.LayerNorm(self.value_dim)

        # Depthwise temporal upsampling:
        # (B, E_v, T_te) -> (B, E_v, T)
        self.temporal_upsampler = nn.ConvTranspose1d(
            in_channels=value_dim,
            out_channels=value_dim,
            kernel_size=temporal_compression_ratio,
            stride=temporal_compression_ratio, # TODO also change it
            groups=value_dim,
            bias=False,
        )

        # # One linear mapping shared over all timepoints:
        # # (B, T, E_v) -> (B, T, n_neurons)
        # self.neural_readout = nn.Linear(
        #     value_dim,
        #     out_dim,
        # ) 

    def train(self, mode=True):
        # Keep the frozen backbone pinned in eval mode regardless of the outer
        # model's mode (now that encoder_backbone is a registered submodule, a
        # plain super().train() call would otherwise re-enable its dropout).
        super().train(mode)
        self.encoder_backbone.eval()
        return self
    # EOF

    @property
    def device(self):
        # Derived from a live parameter rather than the imgANN.device attribute,
        # which stays stale after BaselineModel.to(device) moves the weights.
        return next(self.parameters()).device
    # EOF

    # --- GETTERS ---
    def get_encoder(self):
        return self.encoder

    def get_layer_names(self) -> list[str]:
        return self.layer_names

    def get_encoder_dim(self) -> int:
        return self.encoder_dim

    def get_temporal_embedding_dim(self) -> int:
        return self.temporal_embedding_dim

    def get_n_temporal_embeddings(self) -> int:
        return self.temporal_embeddings.shape[0]

    def get_temporal_compression_ratio(self) -> int:
        return self.temporal_compression_ratio

    def get_n_timepoints(self) -> int:
        return self.n_timepoints

    def get_key_dim(self) -> int:
        return self.key_dim

    def get_query_dim(self) -> int:
        return self.query_dim

    def get_value_dim(self) -> int:
        return self.value_dim

    def get_trainable_parameters(self):
        return (p for p in self.parameters() if p.requires_grad)

    def get_trainable_named_parameters(self):
        return ((name, p) for name, p in self.named_parameters() if p.requires_grad)
    # EOF

    def forward(self, x):
        # extract feats from img encoder 
        # (B H W C) -> (B E L) # E = embedding dimension , L = number of layers
        self.encoder.model(x) # TODO add the features reshaping to match the shape
        layer_features = self.encoder.features
        layer_features = [layer_features[layer] for layer in layer_features.keys()]
        layer_features = torch.stack(layer_features, dim=1)
        print("layer features", layer_features.shape)
        # cross-attention
        # project layers into K V
        # K: (B E L) -> (B E_t L) # E_t = temporal_embedding_dim
        # V: (B E L) -> (B E_v L) # E_v = value_embedding_dim
        # compute attention
        # Q, K, V, E_t -> (B E_v T_te) # T_te = resolution of temporal embeddings

        # Layer features become keys and values
        keys = self.key_projection(layer_features)
        keys = self.key_norm(keys)
        print("keys", keys.shape)
        # (B, L, E_k)

        values = self.value_projection(layer_features)
        values = self.value_norm(values)
        # (B, L, E_v)
        print("values", keys.shape)
        # Temporal embeddings become queries
        if hasattr(self, "query_projection"):
            queries = self.query_projection(self.temporal_embeddings)
        else:
            queries = self.temporal_embeddings
        # end if hasattr(self, "query_projection"):

        # Add batch dimension without creating copies
        queries = queries.unsqueeze(0).expand(layer_features.shape[0], -1, -1) # TODO find a more elegant way to define B
        print("queries", queries.shape)

        # Scaled dot-product attention
        attention_logits = torch.matmul(
            queries,
            keys.transpose(-1, -2),
        )
        # (B, T_te, L)

        attention_logits = attention_logits / math.sqrt(self.key_dim) # Q K.T/sqrt(d)

        attention_weights = torch.softmax(
            attention_logits,
            dim=-1,
        )

        # Softmax across layers L
        # Weighted combination of layer values
        # (B, T_te, L, E_v) -> (B, T_te, E_v)
        coarse_latents = torch.matmul(
            attention_weights,
            values,
        )

        # ConvTranspose1d expects channels-first format
        coarse_latents = coarse_latents.transpose(1, 2)
        # (B, E_v, T_te)

        fine_latents = self.temporal_upsampler(coarse_latents)
        # (B, E_v, T)

        fine_latents = fine_latents.transpose(1, 2)
        # (B, T, E_v)

        return fine_latents, attention_weights
        # (B, T_te, E_v)
        # (T_te, E_k)
        # Cross-attention
        #
        # attention_logits = Q @ K.transpose(-1, -2)
        #
        # attention_logits:
        # (B, T_te, E_k) @ (B, E_k, L)
        # -> (B, T_te, L)
        #
        # attention_weights = softmax(attention_logits / sqrt(E_k), dim=-1)
        #
        # attended_values = attention_weights @ V
        #
        # (B, T_te, L) @ (B, L, E_v)
        # -> (B, T_te, E_v)


        # upsampling (transpose_conv1d)
        # (B E_v T_te) -> (B E_v T)

        # sg(linear mapping)
        # (B E_v T) -> (B d T) # d = number of neurons
        
# 1. Extract features from frozen image encoder
# one pooled feature vector per selected layer
#
# raw features:
# (B, L, E_in)
#
# B    = batch size
# L    = number of selected layers, e.g. 3
# E_in = original backbone embedding dimension


# 2. Project layer features into keys and values
#
# K = key_projection(layer_features)
# V = value_projection(layer_features)
#
# K: (B, L, E_k)
# V: (B, L, E_v)


# 3. Create/project temporal embeddings into queries
#
# temporal_embeddings: (T_te, E_te)
# Q = query_projection(temporal_embeddings)
#
# Q: (T_te, E_k)
#
# expand over batch:
# Q: (B, T_te, E_k)





# 5. Learned temporal upsampling inside the network
#
# (B, T_te, E_v)
# -> (B, T, E_v)
#
# T_te = number of coarse temporal embeddings
# T    = number of neural timepoints, e.g. 30


# 6. One shared linear neural mapping
#
# (B, T, E_v)
# -> (B, T, d)
#
# d = number of neurons

In [8]:
m = BaselineModel(ann, cfg.layer_names, 50, 60, 30, 3).to(ann.device)

In [9]:
for img, neu in loader:
    img = img.to(m.device)
    a = m(img)
    break

layer features torch.Size([8, 3, 1024])
keys torch.Size([8, 3, 50])
values torch.Size([8, 3, 50])
queries torch.Size([8, 10, 50])


In [10]:
a[0].shape

torch.Size([8, 30, 60])